# AcuDock QuickDock - Molecular Docking Pipeline

**Single-click molecular docking powered by Gradio.**

A self-contained notebook that provides an interactive web interface for:
- Fetching & preparing protein targets from PDB
- Preparing ligands from SMILES strings
- Running molecular docking (Vina CPU or Uni-Dock GPU)
- Interactive 3D visualization of binding poses
- Batch screening of compound libraries

## Getting Started

1. **Run the install cell below** (takes ~2-3 minutes)
2. The runtime will **automatically restart** — this is normal
3. After restart, **skip the install cell** and run the remaining cells
4. The Gradio interface will launch inline — interact with it directly!

**No GPU required** for Vina mode. Enable GPU runtime for Uni-Dock acceleration.

---

**License:** MIT | **Platform:** Google Colab | **Author:** AcuDock Project

In [ ]:
# === Step 1: Install Dependencies ===
# After this cell completes, the runtime will restart automatically.
# Once it restarts, SKIP this cell and run the cells below.

!pip install -q vina meeko gemmi rdkit prody py3Dmol openbabel-wheel pdbfixer pandas numpy scipy gradio

# Optional: Install Uni-Dock for GPU-accelerated docking
# Uncomment the lines below if you have a GPU runtime and want 1000x+ speedup
# !pip install -q condacolab
# import condacolab; condacolab.install()
# !mamba install -y -c conda-forge unidock

# Clone AcuDock repo for shared utilities
!git clone https://github.com/Grimlock5310/AcuDock.git /content/AcuDock 2>/dev/null || echo "Repo already cloned"

# Restart runtime so C-extension packages are loadable
import os
os.kill(os.getpid(), 9)

## Launch AcuDock QuickDock

Run the cell below to start the interactive docking interface.
A Gradio app will appear inline with tabs for single docking and batch screening.

In [ ]:
# === Step 2: Imports and Setup ===
import warnings
warnings.filterwarnings('ignore')

import os, sys, io
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

sys.path.insert(0, '/content/AcuDock')
import acudock_utils as utils

WORK_DIR = '/content/acudock_quickdock'
os.makedirs(WORK_DIR, exist_ok=True)

print('Imports successful.')
print(utils.get_docking_engine_status())

In [ ]:
# === Step 3: Launch Gradio Interface ===
import gradio as gr

# ---------------------------------------------------------------------------
# Pipeline functions
# ---------------------------------------------------------------------------

def run_single_docking(pdb_id, smiles, lig_name, engine, exhaustiveness,
                        n_poses, box_size, residues_str, progress=gr.Progress()):
    """Full single-ligand docking pipeline."""
    log_lines = []
    def log(msg):
        log_lines.append(msg)

    try:
        pdb_id = pdb_id.strip().upper()
        smiles = smiles.strip()
        lig_name = lig_name.strip() or 'ligand'

        if not pdb_id:
            return 'Error: Please enter a PDB ID.', None, '', None, None
        if not smiles:
            return 'Error: Please enter a SMILES string.', None, '', None, None
        mol_check = Chem.MolFromSmiles(smiles)
        if mol_check is None:
            return 'Error: Invalid SMILES string.', None, '', None, None

        # Step 1: Prepare protein
        progress(0.1, desc='Preparing protein...')
        log(f'Fetching and preparing protein: {pdb_id}')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)
        log(f'  Protein prepared: {os.path.basename(protein_pdb)}')

        # Step 2: Prepare ligand
        progress(0.25, desc='Preparing ligand...')
        log(f'Preparing ligand: {lig_name}')
        ligand_pdbqt, lig_mol = utils.prepare_ligand(
            smiles, name=lig_name, output_dir=WORK_DIR
        )
        props = utils.get_ligand_properties(smiles)
        log(f'  MW={props["MW"]} LogP={props["LogP"]} HBD={props["HBD"]} HBA={props["HBA"]}')

        # Step 3: Calculate binding site center
        progress(0.35, desc='Defining search box...')
        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip()]
        center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        box = [int(box_size)] * 3
        log(f'  Search box center: [{center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f}]')
        log(f'  Search box size: {box}')

        # Step 4: Run docking
        progress(0.4, desc='Running docking...')
        use_unidock = 'Uni-Dock' in engine and utils.check_unidock_available()

        if use_unidock:
            log(f'Running Uni-Dock GPU docking...')
            energies, poses_path = utils.run_unidock_single(
                receptor_pdbqt, ligand_pdbqt,
                center=center, box_size=box,
                num_modes=int(n_poses), output_dir=WORK_DIR
            )
        else:
            if 'Uni-Dock' in engine:
                log('Uni-Dock not available, falling back to Vina CPU.')
            log(f'Running Vina docking (exhaustiveness={int(exhaustiveness)})...')
            _, energies_raw, poses_path = utils.run_vina(
                receptor_pdbqt, ligand_pdbqt,
                center=center, box_size=box,
                exhaustiveness=int(exhaustiveness), n_poses=int(n_poses)
            )
            energies = [(e[0], e[1], e[2]) for e in energies_raw]

        progress(0.8, desc='Building results...')
        log(f'Docking complete! {len(energies)} poses generated.')
        log(f'Best score: {energies[0][0]:.2f} kcal/mol')
        log(f'Interpretation: {utils.score_interpretation(energies[0][0])}')

        # Build results table
        R, T = 1.987e-3, 298.15
        results_df = pd.DataFrame({
            'Pose': range(1, len(energies) + 1),
            'Score (kcal/mol)': [e[0] for e in energies],
            'RMSD_lb (A)': [round(e[1], 2) for e in energies],
            'RMSD_ub (A)': [round(e[2], 2) for e in energies],
            'Est. Kd (uM)': [round(np.exp(e[0] / (R * T)) * 1e6, 4) for e in energies],
        })

        # Build 3D viewer
        progress(0.9, desc='Building 3D viewer...')
        with open(protein_pdb, 'r') as f:
            prot_data = f.read()
        top_pose = utils.extract_pose_from_pdbqt(poses_path, 0)
        viewer_html = utils.make_3d_viewer_html(prot_data, top_pose)

        # Save CSV
        csv_path = os.path.join(WORK_DIR, f'{lig_name}_{pdb_id}_results.csv')
        results_df.to_csv(csv_path, index=False)

        # 2D structure image
        mol_2d = Chem.MolFromSmiles(smiles)
        img = Draw.MolToImage(mol_2d, size=(300, 200))
        img_buf = io.BytesIO()
        img.save(img_buf, format='PNG')
        img_buf.seek(0)
        img_path = os.path.join(WORK_DIR, f'{lig_name}_2d.png')
        with open(img_path, 'wb') as f:
            f.write(img_buf.read())

        progress(1.0, desc='Done!')
        return '\n'.join(log_lines), results_df, viewer_html, csv_path, img_path

    except Exception as e:
        log(f'ERROR: {str(e)}')
        return '\n'.join(log_lines), None, '', None, None


def run_batch_screening(pdb_id, compounds_text, engine, exhaustiveness,
                         box_size, residues_str, progress=gr.Progress()):
    """Batch screening pipeline."""
    log_lines = []
    def log(msg):
        log_lines.append(msg)

    try:
        pdb_id = pdb_id.strip().upper()
        if not pdb_id:
            return 'Error: Please enter a PDB ID.', None, None, None

        # Parse compound library from text
        compounds = []
        for line in compounds_text.strip().split('\n'):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split(',', 1) if ',' in line else line.split('\t', 1)
            if len(parts) == 2:
                name, smi = parts[0].strip(), parts[1].strip()
            else:
                smi = parts[0].strip()
                name = f'Compound_{len(compounds)+1}'
            if Chem.MolFromSmiles(smi) is not None:
                compounds.append((name, smi))

        if not compounds:
            return 'Error: No valid compounds found.', None, None, None

        log(f'Parsed {len(compounds)} valid compounds.')

        # Prepare protein
        progress(0.05, desc='Preparing protein...')
        log(f'Preparing protein: {pdb_id}')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)

        # Search box
        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip()]
        center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        box = [int(box_size)] * 3

        # Dock each compound
        results = []
        use_unidock = 'Uni-Dock' in engine and utils.check_unidock_available()
        batch_exh = min(int(exhaustiveness), 16)  # Lower for batch speed

        for i, (name, smi) in enumerate(compounds):
            pct = 0.1 + 0.85 * (i / len(compounds))
            progress(pct, desc=f'Docking {i+1}/{len(compounds)}: {name}')
            try:
                lig_pdbqt, _ = utils.prepare_ligand(smi, name=name, output_dir=WORK_DIR)

                if use_unidock:
                    en, _ = utils.run_unidock_single(
                        receptor_pdbqt, lig_pdbqt,
                        center=center, box_size=box,
                        num_modes=5, output_dir=WORK_DIR
                    )
                    best_score = en[0][0] if en else None
                else:
                    _, en, _ = utils.run_vina(
                        receptor_pdbqt, lig_pdbqt,
                        center=center, box_size=box,
                        exhaustiveness=batch_exh, n_poses=5
                    )
                    best_score = en[0][0] if len(en) > 0 else None

                mol_props = utils.get_ligand_properties(smi)
                R, T = 1.987e-3, 298.15
                est_kd = round(np.exp(best_score / (R * T)) * 1e6, 4) if best_score else None
                results.append({
                    'Rank': 0, 'Name': name, 'SMILES': smi,
                    'Score (kcal/mol)': round(best_score, 2) if best_score else None,
                    'Est. Kd (uM)': est_kd,
                    'MW': mol_props.get('MW'),
                    'LogP': mol_props.get('LogP'),
                    'Interpretation': utils.score_interpretation(best_score) if best_score else 'Failed',
                })
                log(f'  [{i+1}/{len(compounds)}] {name}: {best_score:.2f} kcal/mol')
            except Exception as e:
                log(f'  [{i+1}/{len(compounds)}] {name}: FAILED ({e})')
                results.append({
                    'Rank': 0, 'Name': name, 'SMILES': smi,
                    'Score (kcal/mol)': None, 'Est. Kd (uM)': None,
                    'MW': None, 'LogP': None, 'Interpretation': 'Failed',
                })

        # Build results DataFrame
        batch_df = pd.DataFrame(results)
        batch_df = batch_df.sort_values('Score (kcal/mol)', ascending=True).reset_index(drop=True)
        batch_df['Rank'] = range(1, len(batch_df) + 1)
        cols = ['Rank'] + [c for c in batch_df.columns if c != 'Rank']
        batch_df = batch_df[cols]

        log(f'\nBatch complete. Best: {batch_df["Score (kcal/mol)"].min():.2f} kcal/mol')

        # Save CSV
        csv_path = os.path.join(WORK_DIR, f'batch_{pdb_id}_results.csv')
        batch_df.to_csv(csv_path, index=False)

        # Score bar chart
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        valid = batch_df.dropna(subset=['Score (kcal/mol)'])
        fig, ax = plt.subplots(figsize=(10, max(3, len(valid) * 0.4)))
        colors = ['#2ecc71' if s <= -7 else '#f39c12' if s <= -5 else '#e74c3c'
                  for s in valid['Score (kcal/mol)']]
        ax.barh(valid['Name'], valid['Score (kcal/mol)'], color=colors, edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Docking Score (kcal/mol)')
        ax.set_title(f'Batch Screening: {pdb_id}')
        ax.axvline(x=-7, color='gray', linestyle='--', alpha=0.5, label='Moderate threshold')
        ax.legend()
        ax.invert_yaxis()
        plt.tight_layout()
        chart_path = os.path.join(WORK_DIR, 'batch_chart.png')
        fig.savefig(chart_path, dpi=120, bbox_inches='tight')
        plt.close(fig)

        progress(1.0, desc='Done!')
        return '\n'.join(log_lines), batch_df, chart_path, csv_path

    except Exception as e:
        log(f'ERROR: {str(e)}')
        return '\n'.join(log_lines), None, None, None


# ---------------------------------------------------------------------------
# Gradio Interface
# ---------------------------------------------------------------------------

DEFAULT_COMPOUNDS = """Aspirin, CC(=O)Oc1ccccc1C(=O)O
Ibuprofen, CC(C)Cc1ccc(cc1)C(C)C(=O)O
Caffeine, Cn1c(=O)c2c(ncn2C)n(C)c1=O
Acetaminophen, CC(=O)Nc1ccc(O)cc1
Naproxen, COc1ccc2cc(ccc2c1)C(C)C(=O)O
Metformin, CN(C)C(=N)NC(=N)N
Celecoxib, Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1
Diclofenac, OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl"""

with gr.Blocks(
    title='AcuDock QuickDock',
    theme=gr.themes.Soft(),
    css='.gradio-container { max-width: 1200px !important; }'
) as demo:

    gr.Markdown("""
    # AcuDock QuickDock
    **Molecular docking made simple.** Enter a protein PDB ID and a ligand SMILES to dock.
    """)

    with gr.Tabs():

        # === Single Docking Tab ===
        with gr.Tab('Single Docking'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### Target & Ligand')
                    pdb_input = gr.Textbox(label='PDB ID', value='1HSG',
                                           placeholder='e.g. 1HSG, 4LDE, 6LU7')
                    smiles_input = gr.Textbox(
                        label='Ligand SMILES',
                        value='CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCN2C(=O)[C@H](CC2=CC=CC=C2)NC(=O)[C@@H](CC2=CC=C(O)C=C2)N1',
                        lines=2
                    )
                    name_input = gr.Textbox(label='Ligand Name', value='Indinavir')

                    gr.Markdown('### Docking Parameters')
                    engine_input = gr.Radio(
                        ['Vina (CPU)', 'Uni-Dock (GPU)'],
                        value='Vina (CPU)', label='Docking Engine'
                    )
                    exhaust_input = gr.Slider(8, 128, value=32, step=8,
                                              label='Exhaustiveness (Vina only)')
                    poses_input = gr.Slider(5, 50, value=20, step=5,
                                            label='Max Poses')
                    box_input = gr.Slider(15, 40, value=20, step=5,
                                          label='Box Size (Angstrom)')
                    residues_input = gr.Textbox(
                        label='Active Site Residues (comma-separated)',
                        value='23,24,25,26,27,28,29,30',
                        placeholder='Leave blank to use protein centroid'
                    )
                    dock_btn = gr.Button('Run Docking', variant='primary', size='lg')

                with gr.Column(scale=2):
                    log_output = gr.Textbox(label='Log', lines=8, interactive=False)
                    ligand_img = gr.Image(label='Ligand 2D Structure', height=200)
                    results_table = gr.Dataframe(label='Docking Results')
                    viewer_output = gr.HTML(label='3D Viewer')
                    csv_output = gr.File(label='Download Results CSV')

            dock_btn.click(
                fn=run_single_docking,
                inputs=[pdb_input, smiles_input, name_input, engine_input,
                        exhaust_input, poses_input, box_input, residues_input],
                outputs=[log_output, results_table, viewer_output, csv_output, ligand_img]
            )

        # === Batch Screening Tab ===
        with gr.Tab('Batch Screening'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### Batch Configuration')
                    gr.Markdown('Use the PDB ID and active site from the Single Docking tab.')
                    batch_pdb = gr.Textbox(label='PDB ID', value='1HSG')
                    batch_residues = gr.Textbox(
                        label='Active Site Residues',
                        value='23,24,25,26,27,28,29,30'
                    )
                    batch_engine = gr.Radio(
                        ['Vina (CPU)', 'Uni-Dock (GPU)'],
                        value='Vina (CPU)', label='Engine'
                    )
                    batch_exhaust = gr.Slider(8, 64, value=8, step=8,
                                              label='Exhaustiveness')
                    batch_box = gr.Slider(15, 40, value=20, step=5,
                                          label='Box Size (Angstrom)')
                    compounds_input = gr.Textbox(
                        label='Compound Library (Name, SMILES per line)',
                        value=DEFAULT_COMPOUNDS,
                        lines=10,
                        placeholder='Name, SMILES (one per line)'
                    )
                    batch_btn = gr.Button('Run Batch Screening', variant='primary', size='lg')

                with gr.Column(scale=2):
                    batch_log = gr.Textbox(label='Log', lines=8, interactive=False)
                    batch_table = gr.Dataframe(label='Batch Results')
                    batch_chart = gr.Image(label='Score Chart')
                    batch_csv = gr.File(label='Download Batch CSV')

            batch_btn.click(
                fn=run_batch_screening,
                inputs=[batch_pdb, compounds_input, batch_engine,
                        batch_exhaust, batch_box, batch_residues],
                outputs=[batch_log, batch_table, batch_chart, batch_csv]
            )

        # === About Tab ===
        with gr.Tab('About'):
            gr.Markdown("""
            ### AcuDock QuickDock

            A streamlined molecular docking pipeline for computational drug discovery.

            **Pipeline:**
            ```
            PDB ID --> PDBFixer --> PDBQT (receptor)
            SMILES --> RDKit 3D --> Meeko --> PDBQT (ligand)
                                               |
                           Vina / Uni-Dock Docking
                                               |
                                      Ranked Poses + 3D Viewer
            ```

            **Score Interpretation:**
            | Score (kcal/mol) | Binding | Approx. Kd |
            |:---:|---|---|
            | > -5 | Very weak | > 100 uM |
            | -5 to -7 | Moderate | 10-100 uM |
            | -7 to -9 | Good | 100 nM - 10 uM |
            | < -9 | Strong | < 100 nM |

            **Docking Engines:**
            - **Vina (CPU):** AutoDock Vina, 90.2% CASF docking power. No GPU needed.
            - **Uni-Dock (GPU):** 1000x+ speedup on NVIDIA GPUs. Requires GPU runtime.

            **Tips:**
            - Validate by redocking a co-crystallized ligand (RMSD < 2A = success)
            - Preparation quality matters more than algorithm choice
            - Vina scores have ~2 kcal/mol error margin

            *MIT License | AcuDock Project*
            """)

demo.launch()